In [1]:
# Q1
import pandas as pd

roll_number = "1024170388"

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"},
]

last_two_digits = [int(ch) for ch in roll_number[-2:]]
categories = ["billing", "account", "general"]

personalized_entries = [
    {"question": "where is your office located", "answer": "Our office is located at Block A, Main Campus.",
     "keywords": "location address office", "category": categories[last_two_digits[0] % 3]},
    {"question": "what documents are required for admission", "answer": "You need your ID proof, previous mark sheets, and passport size photos.",
     "keywords": "documents required admission", "category": categories[last_two_digits[1] % 3]},
]

faq_entries = fixed_entries + personalized_entries
df = pd.DataFrame(faq_entries)
df

,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,where is your office located,"Our office is located at Block A, Main Campus.",location address office,general
5,what documents are required for admission,"You need your ID proof, previous mark sheets, ...",documents required admission,general


In [2]:
# Q2
def score_query(query, df):
    query_words = set(query.lower().split())
    scores = []
    for i, row in df.iterrows():
        keyword_words = set(row["keywords"].lower().split())
        score = len(query_words & keyword_words)
        scores.append(score)
    result_df = df.copy()
    result_df["confidence"] = scores
    result_df = result_df[result_df["confidence"] > 0].sort_values("confidence", ascending=False)
    return result_df

score_query("how do i pay the fee", df)

,question,answer,keywords,category,confidence
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing,2
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing,1


In [3]:
# Q3
def same_category(category_name, df):
    return df[df["category"] == category_name]

print(same_category("general", df))

                                    question  \
2                what are your working hours   
4               where is your office located   
5  what documents are required for admission   

                                              answer  \
2                          We are open 9 AM to 5 PM.   
4     Our office is located at Block A, Main Campus.   
5  You need your ID proof, previous mark sheets, ...   

                       keywords category  
2        hours timing open time  general  
4       location address office  general  
5  documents required admission  general  


In [4]:
# Q4
entry_index = 0
new_keyword = input("Enter a new keyword to add: ")
df.loc[entry_index, "keywords"] = df.loc[entry_index, "keywords"] + " " + new_keyword

output_filename = f"{roll_number}_faq_data.csv"
df.to_csv(output_filename, index=False)
df

Enter a new keyword to add:  charges


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge charges,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,where is your office located,"Our office is located at Block A, Main Campus.",location address office,general
5,what documents are required for admission,"You need your ID proof, previous mark sheets, ...",documents required admission,general


In [5]:
# Q5
print(df.groupby("category").size())

category
account    1
billing    2
general    3
dtype: int64


In [6]:
# Q6
def score_query_v2(query, df):
    query_words = set(query.lower().split())
    scores = []
    for i, row in df.iterrows():
        keyword_words = set(row["keywords"].lower().split())
        score = len(query_words & keyword_words)
        scores.append(score)
    result_df = df.copy()
    result_df["confidence"] = scores
    matched = result_df[result_df["confidence"] > 0]

    if matched.empty:
        print("No matching entries found.")
        return None

    max_score = matched["confidence"].max()
    top_matches = matched[matched["confidence"] == max_score]

    if len(top_matches) > 1:
        print(f"Tie detected between {len(top_matches)} entries with confidence {max_score}:")
    else:
        print(f"Best match found with confidence {max_score}:")

    print(top_matches)
    return top_matches

In [7]:
score_query_v2("fee", df)

Tie detected between 2 entries with confidence 1:
                 question                                      answer  \
0  what is the annual fee                   The annual fee is Rs 500.   
3   how can i pay the fee  You can pay via UPI, card, or net banking.   

                        keywords category  confidence  
0  fee cost price charge charges  billing           1  
3            pay payment upi fee  billing           1  


,question,answer,keywords,category,confidence
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge charges,billing,1
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing,1


In [8]:
score_query_v2("reset password", df)

Best match found with confidence 2:
                question                            answer  \
1  how to reset password  Go to Settings > Reset Password.   

               keywords category  confidence  
1  password reset login  account           2  


,question,answer,keywords,category,confidence
1,how to reset password,Go to Settings > Reset Password.,password reset login,account,2
